In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Step 1: Load and Preprocess
path = r"C:\Users\Girija S.H\.cache\kagglehub\datasets\prachi13\customer-analytics\versions\1"
train_path = os.path.join(path, "Train.csv")
df = pd.read_csv(train_path)

# Drop ID
df = df.drop(columns=["ID"])

# Separate features and target
X = df.drop(columns=["Reached.on.Time_Y.N"])
y = df["Reached.on.Time_Y.N"]

# One-hot encode categoricals
X = pd.get_dummies(X, drop_first=True)

# Scale numerical features
scaler = StandardScaler()
X[X.columns] = scaler.fit_transform(X)

# Convert to numpy arrays
X = X.values
y = y.values

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Step 2: SVM with RBF Kernel + GridSearch
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': [0.01, 0.1, 1, 'scale'],
    'kernel': ['rbf']
}

grid = GridSearchCV(SVC(), param_grid, refit=True, verbose=2, cv=5, n_jobs=-1)
grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)

# Predict using best model
y_pred = grid.predict(X_test)

# Step 3: Evaluation
accuracy = accuracy_score(y_test, y_pred)
print("\nAccuracy (Best RBF SVM):", accuracy)
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Step 1: Load and Preprocess
path = r"C:\Users\Girija S.H\.cache\kagglehub\datasets\prachi13\customer-analytics\versions\1"
train_path = os.path.join(path, "Train.csv")
df = pd.read_csv(train_path)

# Drop ID
df = df.drop(columns=["ID"])

# Separate features and target
X = df.drop(columns=["Reached.on.Time_Y.N"])
y = df["Reached.on.Time_Y.N"]

# One-hot encode categoricals
X = pd.get_dummies(X, drop_first=True)

# Scale numerical features
scaler = StandardScaler()
X[X.columns] = scaler.fit_transform(X)

# Convert to numpy arrays
X = X.values
y = y.values.reshape(-1, 1)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Step 2: SVM From Scratch
class SVM_Scratch:
    def __init__(self, lr=0.001, lambda_param=0.01, n_iters=1000):
        self.lr = lr
        self.lambda_param = lambda_param
        self.n_iters = n_iters
        self.w = None
        self.b = None

    def fit(self, X, y):
        n_samples, n_features = X.shape
        # Convert labels from {0,1} to {-1,1}
        y_ = np.where(y <= 0, -1, 1).flatten()

        self.w = np.zeros(n_features)
        self.b = 0

        for _ in range(self.n_iters):
            for idx, x_i in enumerate(X):
                condition = y_[idx] * (np.dot(x_i, self.w) - self.b) >= 1
                if condition:
                    # Gradient when correctly classified
                    self.w -= self.lr * (2 * self.lambda_param * self.w)
                else:
                    # Gradient when misclassified
                    self.w -= self.lr * (2 * self.lambda_param * self.w - np.dot(x_i, y_[idx]))
                    self.b -= self.lr * y_[idx]

    def predict(self, X):
        linear_output = np.dot(X, self.w) - self.b
        return np.where(linear_output >= 0, 1, 0)

# Step 3: Train & Evaluate
svm_scratch = SVM_Scratch(lr=0.001, lambda_param=0.01, n_iters=1000)
svm_scratch.fit(X_train, y_train)

y_pred_scratch_svm = svm_scratch.predict(X_test)

accuracy_scratch_svm = np.mean(y_pred_scratch_svm == y_test.ravel())
print("Accuracy (Scratch SVM):", accuracy_scratch_svm)
